# 📉 Phase 3: Liquidity Stress-Test Engine

## 🎯 Business Objective
Market volatility (Phase 2) is only a risk if it impacts the bottom line. In this phase, we translate freight rate shocks into financial KRI impacts. 

We will model a mid-sized logistics entity and stress-test its liquidity runway under varying freight cost shocks. 
*Crucially, in the severe scenario, we will also model a deterioration in Days Sales Outstanding (DSO), simulating a compound crisis where clients delay payments just as costs spike.*

In [1]:
import pandas as pd
import numpy as np

# Load the master dataset we created in Phase 2
df = pd.read_csv('freight_data.csv')
df['Date'] = pd.to_datetime(df['Date'])

print(f"✅ Loaded {len(df)} days of freight data for stress testing.")

✅ Loaded 784 days of freight data for stress testing.


## 📊 Step 1: Advanced Risk Metrics (Volatility & Z-Scores)
Before running the stress test, we need to establish our risk thresholds. We will calculate:
1. **Rolling 30-Day Annualized Volatility:** To track market turbulence.
2. **Z-Score:** To identify statistically abnormal rate spikes (e.g., a Z-score > 2.0 means rates are 2 standard deviations above the norm).

In [2]:
# Calculate daily returns
df['SCFI_Returns'] = df['SCFI'].pct_change()

# 30-Day Rolling Annualized Volatility
df['SCFI_Vol_30D_Ann'] = df['SCFI_Returns'].rolling(window=30).std() * np.sqrt(252)

# 60-Day Rolling Z-Score (How far is the current rate from the 60-day moving average?)
df['SCFI_MA_60'] = df['SCFI'].rolling(window=60).mean()
df['SCFI_Std_60'] = df['SCFI'].rolling(window=60).std()
df['SCFI_Z_Score'] = (df['SCFI'] - df['SCFI_MA_60']) / df['SCFI_Std_60']

# Define Risk Appetite Thresholds (Example corporate limits)
VOLATILITY_LIMIT = 0.50 # 50% annualized vol
Z_SCORE_LIMIT = 2.0     # 2 standard deviations

# Flag breaches
df['Vol_Breach'] = df['SCFI_Vol_30D_Ann'] > VOLATILITY_LIMIT
df['Z_Breach'] = df['SCFI_Z_Score'] > Z_SCORE_LIMIT

print(f"⚠️ Volatility Limit Breached: {df['Vol_Breach'].sum()} days")
print(f"⚠️ Z-Score Limit Breached: {df['Z_Breach'].sum()} days")

⚠️ Volatility Limit Breached: 31 days
⚠️ Z-Score Limit Breached: 31 days


## 🧮 Step 2: The Liquidity Stress-Test Model
We will now model the financial impact. 

**Baseline Assumptions (in $ Millions):**
- **Monthly Revenue:** $50M
- **Freight Costs:** 40% of revenue ($20M) - *highly sensitive to SCFI*
- **Fixed OPEX:** $10M (salaries, leases, etc.)
- **Liquidity Buffer:** $60M cash on hand

**The Scenarios:**
1. **Base Case:** 0% freight cost increase.
2. **Mild Shock:** 10% freight cost increase (e.g., minor port congestion).
3. **Severe Shock:** 30% freight cost increase (e.g., major geopolitical disruption).
4. **Compound Crisis (The "Actuarial" Scenario):** 30% freight shock + **DSO deterioration** (clients delay payments by 15 days, freezing working capital).

In [3]:
# --- REALISTIC Baseline Financials (in $ Millions) ---
MONTHLY_REVENUE = 50.0
FREIGHT_COST_PCT = 0.60 # Freight is usually 60% of revenue in logistics
BASE_FREIGHT_COST = MONTHLY_REVENUE * FREIGHT_COST_PCT # $30M
FIXED_OPEX = 15.0 # $15M (warehouses, staff, tech)
LIQUIDITY_BUFFER = 30.0 # $30M cash (smaller buffer = higher risk)
BASE_DSO = 45 

def run_liquidity_stress_test(shock_pct, dso_increase_days=0):
    shocked_freight_cost = BASE_FREIGHT_COST * (1 + shock_pct)
    total_monthly_costs = shocked_freight_cost + FIXED_OPEX
    monthly_net_cash_flow = MONTHLY_REVENUE - total_monthly_costs
    
    daily_revenue = MONTHLY_REVENUE / 30
    cash_trapped_in_receivables = daily_revenue * dso_increase_days
    effective_buffer = LIQUIDITY_BUFFER - cash_trapped_in_receivables
    
    if monthly_net_cash_flow >= 0:
        runway_months = 99.0 # Cap visual at 99 for "Profitable"
        risk_flag = "SAFE 🟢"
    else:
        monthly_burn_rate = abs(monthly_net_cash_flow)
        runway_months = effective_buffer / monthly_burn_rate
        if runway_months < 3:
            risk_flag = "CRITICAL 🔴"
        elif runway_months < 6:
            risk_flag = "WARNING 🟠"
        else:
            risk_flag = "CAUTION 🟡"
            
    return {
        'Scenario': f"{int(shock_pct*100)}% Freight Shock",
        'DSO Impact': f"+{dso_increase_days} days" if dso_increase_days > 0 else "None",
        'Monthly Freight Cost ($M)': round(shocked_freight_cost, 2),
        'Monthly Cash Flow ($M)': round(monthly_net_cash_flow, 2),
        'Cash Trapped in AR ($M)': round(cash_trapped_in_receivables, 2),
        'Effective Buffer ($M)': round(effective_buffer, 2),
        'Liquidity Runway (Months)': round(runway_months, 1),
        'Risk Flag': risk_flag
    }

# Run the 4 Scenarios
scenarios = [
    run_liquidity_stress_test(0.00, 0),   
    run_liquidity_stress_test(0.10, 0),   
    run_liquidity_stress_test(0.30, 0),   
    run_liquidity_stress_test(0.30, 15)   # The Compound Crisis
]

df_stress = pd.DataFrame(scenarios)

print("-" * 60)
print("🎉 PHASE 3 COMPLETE! Liquidity Stress-Test Results:")
print("-" * 60)
display(df_stress)

df_stress.to_csv('stress_test_results.csv', index=False)

------------------------------------------------------------
🎉 PHASE 3 COMPLETE! Liquidity Stress-Test Results:
------------------------------------------------------------


,Scenario,DSO Impact,Monthly Freight Cost ($M),Monthly Cash Flow ($M),Cash Trapped in AR ($M),Effective Buffer ($M),Liquidity Runway (Months),Risk Flag
0,0% Freight Shock,None,30.0,5.0,0.0,30.0,99.0,SAFE 🟢
1,10% Freight Shock,None,33.0,2.0,0.0,30.0,99.0,SAFE 🟢
2,30% Freight Shock,None,39.0,-4.0,0.0,30.0,7.5,CAUTION 🟡
3,30% Freight Shock,+15 days,39.0,-4.0,25.0,5.0,1.2,CRITICAL 🔴
